# 20 — fitting a real measurement

Notebook 19 read CBM56 variant 3 and said what is in it. This one fits it, and
reports the distance distribution, the weighted residuals and every nuisance
against its prior.

It also records what had to change, and what broke, because three of the four
changes were only needed once the data came from an instrument rather than from
the model itself.


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import cbm56
import experiment as X


## 1. The model's time axis

The transfer maps are functions of the time axis, and the prototype's is a 50 ns
period at 32 ps because that is the instrument it was written against. CBM56 is
31.25 ns at 64 ps. Both are arguments to the builder, so this is a parameter
change rather than a redesign — but one that would be invisible if it were wrong.

**A choice inside it.** 488 channels of exactly 64 ps is 31.232 ns against the
calibration file's 31.24922561, so either the period is 0.055 % short or every
channel is 0.055 % wide. The TAC calibration is kept exact, because it is what a
lifetime is measured in, and 0.055 % of a wrap-around is a smaller error than
0.055 % of a lifetime.


In [ ]:
t0 = time.time()
m = cbm56.model(samples=('D0', 'DA'), detectors=('gp', 'gs'))
print(f'{time.time() - t0:.0f} s')
E = m['E']
print(f'axis   {E["n"]} channels of {E["dt"] * 1000:.1f} ps  ->  {E["period"]:.4f} ns period')
print(f'       the calibration file says {m["cal"]["period"]:.5f} ns '
      f'({100 * (m["cal"]["period"] / E["period"] - 1):.3f} % longer)')
print(f'basis  {E["K"]} columns over {E["Kint"]} lifetimes; distance grid {len(m["rel"])} points')
print(f'graph  {len(m["keys"])} physics channels, {len(m["graph"].data_keys)} histograms, '
      f'{m["graph"].dim} free coordinates')


### Why only the green detectors here

This notebook fits the **donor's own decay**: the donor-only sample and the FRET
sample, in the two green detectors, under both pulse windows. The distance comes
from how much the donor is quenched in the FRET sample relative to the reference,
which is the classical donor-decay FRET analysis and is the geometry notebook 12
calls donor excitation only.

The acceptor channels are left out, and that is not tidiness. A free
reconvolution of the red-detector histograms — the best any response can do —
cannot describe them, while the same test on the green ones reaches a deviance
per degree of freedom near one. Something about the red detectors' response is
wrong, and until it is found, including those histograms would let a bad
response contaminate a distance. What is missing because of it is stated at the
end.


## 2. The instrument response, and which pulse it comes from

The green detectors see only the green pulse, so there is no choice. The red
detectors see both, and their green-pulse response is far the worse measured —
about 2,250 counts at its peak against a pedestal of 710 per channel.


In [ ]:
info = m['irf_info']
print(f'{"detector":<10}{"taken from":<28}{"peak ch":>8}{"peak":>9}{"flat/ch":>9}{"background":>12}')
for det, v in info.items():
    print(f'{det:<10}{v["source"]:<28}{v["peak_channel"]:>8d}{v["peak"]:>9,.0f}'
          f'{v["flat_per_channel"]:>9.0f}{v["background_fraction"]:>11.1%}')
print(f'\nthe red pulse is {m["offset_channels"]} channels '
      f'({m["offset_channels"] * m["cal"]["dt"]:.3f} ns) behind the green one')


In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 3.6))
for det, r in m['irf'].items():
    ax.semilogy(np.arange(len(r)) * m['cal']['dt'], np.maximum(r, 0.5), lw=0.9, label=det)
ax.set_xlabel('time / ns'); ax.set_ylabel('counts per channel')
ax.set_title('the measured responses used here', fontsize=9)
ax.set_xlim(0, 14); ax.legend(fontsize=7)
fig.tight_layout(); plt.show()


## 3. The response's own background, as a parameter

A measured response is a histogram like any other and carries dark counts,
afterpulsing and room light. Subtracting a number somebody chose puts an
unchecked constant into the shape every lifetime column is built from; fitting
it does not, and that is what `irf_bg_<detector>` is.

**It is only weakly identifiable, and the reason is worth stating rather than
hoping.** A pedestal convolved with a decay is a constant, and the basis already
has a flat column, so to first order the two are the same thing. What separates
them is the normalisation: every column is scaled to unit sum, so the pedestal's
share of a column depends on that column's lifetime and enters as a *different*
amount of flat in each. Whether that is enough here is answered by the posterior
further down, not by this paragraph.

### The bug this introduced, and what hid it

The first version spread the background over every channel of the window. A
measured response isolated to one pulse is exactly **zero outside its own
support**, so a third of the requested background was subtracted from channels
that never had any, and the parameter did not mean the fraction its name claims.

The simulated response could not have shown it. It is analytic, unit-sum and
non-zero everywhere, so the channel count and the support are the same number
and the normalisation is right by accident. The cell below is the check that now
guards it.


In [ ]:
import torch
L = m['L']
n = E['n']
an = torch.tensor(np.exp(-0.5 * ((np.arange(n) - 48) / 2.3) ** 2)); an = an / an.sum()
im = L.InstrumentModel(E, 'measured', {'g': an})
print('an analytic, unit-sum, full-support response with no background node:')
print('   unchanged  ', bool(torch.allclose(im._drop_background(an, {}, 'g'), an)))

core = np.exp(-0.5 * ((np.arange(n) - 48) / 2.3) ** 2) * 30000.0
ped, a, b = 80.0, 33, 200
h = np.zeros(n); h[a:b] = core[a:b] + ped
frac = ped * (b - a) / h.sum()
out = im._drop_background(torch.tensor(h), {'irf_bg_g': torch.tensor([frac])}, 'g').numpy()
print(f'a windowed response with a known pedestal of {ped:.0f} per channel over {b - a} channels:')
print(f'   the flat region reads {out[150:199].mean():.3f} per channel afterwards')
print(f'   outside the window it stays zero: {bool(np.all(out[:a] == 0) and np.all(out[b:] == 0))}')
print(f'   counts kept {out.sum():,.0f} of {h.sum():,.0f}, expected {h.sum() - ped * (b - a):,.0f}')


In [ ]:
print('the starting value for each, measured from the flat part of the water histogram:')
for det, v in m['irf_bg_medians'].items():
    print(f'   irf_bg_{det}   {v:.3f}')


## The fit

Both pulse windows in each histogram, the roughness weight of the distance
prior integrated out over a grid of nodes mixed by their evidences.

**Automatic differentiation, not the analytic Jacobian.** The hand-written
amplitude Jacobian covers three physics cases and this measurement has six —
every sample is seen under both pulses — and the prototype raises rather than
quietly returning the wrong derivative. That is why this takes minutes.


In [ ]:
t0 = time.time()
post = cbm56.fit(m, lam_nodes=(1.0, 0.0, -1.0), verbose=False)
print(f'{time.time() - t0:.0f} s')


## Rule 0

The Poisson deviance per degree of freedom against a **measured** reference —
Poisson draws at the fitted means — and a runs test on the weighted residuals of
every histogram. Not against one: the reported degrees of freedom subtract the
whole parameter dimension while a penalised spline uses far fewer.


In [ ]:
rows, ref, z = cbm56.rule0(m, post)
print(f'{"histogram":<16}{"photons":>12}{"D/dof":>9}{"runs p":>9}')
for r in rows:
    print(f'{r["channel"]:<16}{r["counts"]:>12,.0f}{r["dpd"]:>9.3f}{r["runs_p"]:>9.3f}')
print(f'\nreference D/dof: {ref[0]:.3f} +- {ref[1]:.3f}')
print(f'this fit: {post["dev"] / post["dof"]:.3f}, which is {z:+.1f} reference sd')
PASSES = abs(z) < 3
print('VERDICT:', 'passes' if PASSES else 'FAILS - excluded and counted, and the '
      'numbers below are reported as a diagnosis, not as a result')


In [ ]:
ks = list(m['graph'].data_keys)
ncol = min(len(ks), 4); nrow = int(np.ceil(len(ks) / ncol))
fig, axes = plt.subplots(2 * nrow, ncol, figsize=(3.6 * ncol, 3.4 * nrow), squeeze=False,
                         gridspec_kw=dict(height_ratios=[2.2, 1.0] * nrow))
t = np.arange(m['n']) * m['cal']['dt']
for j, k in enumerate(ks):
    r_, c_ = divmod(j, ncol)
    a, ar = axes[2 * r_][c_], axes[2 * r_ + 1][c_]
    yk = np.asarray(m['y'][k]); lk = np.asarray(post['lam'][k])
    a.semilogy(t, np.maximum(yk, 0.3), 'k.', ms=1.2, label='data')
    a.semilogy(t, np.clip(lk, 1e-2, None), 'C3-', lw=0.9, label='fit')
    rr = post['rows'][k]
    a.set_title(f'{k[0]} {k[1]}\nD/dof {rr["dpd"]:.3f}, runs p {rr["runs_p"]:.2f}', fontsize=8)
    a.set_ylim(0.3, 3 * max(yk.max(), 1))
    if j == 0:
        a.legend(fontsize=7)
    ar.plot(t, rr['w'], lw=0.5, color='C3'); ar.axhline(0, color='k', lw=0.5)
    ar.set_ylim(-6, 6); ar.set_xlabel('time / ns')
    if c_ == 0:
        a.set_ylabel('counts'); ar.set_ylabel('w. residual')
fig.tight_layout(); plt.show()


## The distance distribution

In units of the Förster radius, because $R_0$ is not in the folder. The model
works in $R/R_0$ and needs none to fit; converting the axis to Ångström does,
and whoever knows it can multiply.


In [ ]:
rel = m['rel']
mean, lo1, hi1, lo2, hi2 = L.delta_bands(None, post, rel, space='linear')
fig, ax = plt.subplots(figsize=(7.0, 4.0))
ax.fill_between(rel, lo2, hi2, color='C0', alpha=0.18, lw=0, label='posterior, 2 sd')
ax.fill_between(rel, lo1, hi1, color='C0', alpha=0.35, lw=0, label='posterior, 1 sd')
ax.plot(rel, mean, 'C0-', lw=1.7, label='posterior mean')
L.shade_window(ax, 0.01)
ax.set_xlabel('$R/R_0$'); ax.set_ylabel('p per grid point')
ax.set_xlim(rel[0], rel[-1]); ax.set_ylim(0, None)
ax.set_title('CBM56 Var3, from the donor decay' + ('' if PASSES else '  (FIT FAILED - diagnosis only)'),
             fontsize=10)
ax.legend(fontsize=7, frameon=False)
fig.tight_layout(); plt.show()
mu = float((mean * rel).sum())
print(f'mean R/R0 {mu:.3f}, sd {float(np.sqrt((mean * (rel - mu) ** 2).sum())):.3f}')
print('the grey regions are where the transfer efficiency is within 0.01 of zero or one:')
print('no decay can place a distance there, and a curve drawn through them is the prior')


## Every nuisance, against its prior

On a real sample there is no truth to check an answer against, so the honest
companion to a number is what the measurement did **not** determine. The Laplace
approximation gives each nuisance's posterior directly; drawn against its prior,
a bar that fills its grey band is a parameter the data left alone, and the
number in brackets is the fraction of the prior's variance the data removed.


In [ ]:
nu = cbm56.nuisances(m, post)
ax = cbm56.plot_nuisances(nu)
ax.set_title('the nuisances: posterior against prior', fontsize=10)
plt.tight_layout(); plt.show()


In [ ]:
print(f'{"parameter":<18}{"group":<6}{"posterior":>24}{"prior":>24}{"learned":>9}')
for r in sorted(nu, key=lambda r: -r['learned']):
    print(f'{r["name"]:<18}{r["group"]:<6}'
          f'{r["posterior"]:>14.4f} +-{r["posterior_sd"]:<8.4f}'
          f'{r["prior"]:>14.4f} +-{r["prior_sd"]:<8.4f}{r["learned"]:>9.2f}')


In [ ]:
spec = post['graph'].spectrum(post['graph'].unpack(post['theta'])[0]).detach().numpy()
tc = np.asarray(E['tau_c'])
tau_x = float((spec * tc).sum() / max(spec.sum(), 1e-30))
tau_f = float((spec * tc * tc).sum() / max((spec * tc).sum(), 1e-30))
print(f'the donor lifetime spectrum: species-weighted {tau_x:.3f} ns, '
      f'intensity-weighted {tau_f:.3f} ns')
print(f'Alex fitted {m["cal"]["tau_donor"]:.1f} ns with a single exponential; '
      'that is a check, not a target')


## What this establishes, and what it does not

Read the verdict line and the nuisance table before the distance distribution.

**Not established, and it is the largest gap.** The acceptor channels are not in
this fit. A free reconvolution cannot describe the red-detector histograms with
the response taken from their own water measurement, while the same test on the
green ones reaches a deviance per degree of freedom near one, so the red
detectors' response is wrong in a way not yet found. Until it is, the sensitised
emission, the acceptor's anisotropy and the direct-excitation crosstalk are all
outside what this notebook can say, and the distance here rests on the donor's
quenching alone.

The reference dye in `Rh110_thick/` is the obvious next thing to try: the model
already carries the delta-function convolution method for a reference-dye
response (`irf_tauref_<detector>`, Zuker et al. 1985), which needs no
deconvolution of a weak water measurement.

**No comparison with Alex's own analysis**, whose results are not in the folder,
and no repeat on a second sample. And $R_0$ is not in the folder, so the answer
is in $R/R_0$.
